# 02 - Brochure de negocio, sin caja negra

Este notebook construye un brochure desde una web de empresa. Primero muestra prompts y `messages` manuales. Después muestra cómo esos mismos prompts se guardan en `PromptRegistry` para no repetir strings.

## Variables del caso

`COMPANY_NAME` es el nombre que aparecerá en el brochure.

`COMPANY_URL` es la web que se inspecciona.

Hugging Face se usa solo como ejemplo público. No se usa su API ni sus modelos. Puedes reemplazarlo por cualquier empresa.

`RUN_WEB_FETCH` decide si usamos internet real o datos demo.

`RUN_LIVE_LLM_CALLS` decide si llamamos al modelo o usamos respuestas simuladas.

`MODEL_ID` decide el modelo. Si usas `gpt-5-nano`, el provider sale de `.env`.

In [3]:
import json

from IPython.display import Markdown, display

from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory, build_messages
from llmkit.prompts import PromptRegistry
from llmkit.schemas import BrochureLinkSelection, parse_json_output
from llmkit.web import fetch_website_contents, fetch_website_links

get_settings.cache_clear()
settings = get_settings()

COMPANY_NAME = "Hugging Face"
COMPANY_URL = "https://huggingface.co"
RUN_WEB_FETCH = True
RUN_LIVE_LLM_CALLS = False
MODEL_ID = settings.default_model

print("Company:", COMPANY_NAME)
print("URL:", COMPANY_URL)
print("Model:", MODEL_ID)

Company: Hugging Face
URL: https://huggingface.co
Model: gpt-5-nano


## Paso 1: links candidatos

`sample_links` existe solo para que el notebook sea ejecutable sin internet. Si `RUN_WEB_FETCH=True`, se ignora y se usan links reales extraídos desde `COMPANY_URL`.

In [4]:
sample_links = [
    "https://huggingface.co/about",
    "https://huggingface.co/models",
    "https://huggingface.co/datasets",
    "https://huggingface.co/careers",
    "https://huggingface.co/privacy",
]

candidate_links = fetch_website_links(COMPANY_URL) if RUN_WEB_FETCH else sample_links

for link in candidate_links:
    print("-", link)

- https://huggingface.co/
- https://huggingface.co/models
- https://huggingface.co/datasets
- https://huggingface.co/spaces
- https://huggingface.co/storage
- https://huggingface.co/docs
- https://huggingface.co/enterprise
- https://huggingface.co/pricing
- https://huggingface.co/login
- https://huggingface.co/join
- https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro
- https://huggingface.co/moonshotai/Kimi-K2.6
- https://huggingface.co/Qwen/Qwen3.6-27B
- https://huggingface.co/openai/privacy-filter
- https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
- https://huggingface.co/spaces/smolagents/ml-intern
- https://huggingface.co/spaces/r3gm/wan2-2-fp8da-aoti-preview
- https://huggingface.co/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
- https://huggingface.co/spaces/webml-community/bonsai-ternary-webgpu
- https://huggingface.co/spaces/k2-fsa/OmniVoice
- https://huggingface.co/datasets/nvidia/Nemotron-Personas-Korea
- https://huggingface.co/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cl

## Paso 2 manual: seleccionar links relevantes

Esta es la versión explícita. `link_system_prompt` define el criterio del modelo. `link_user_prompt` contiene la web y la lista de links. `link_messages` es lo que se enviaría al proveedor.

In [5]:
link_system_prompt = (
    "You inspect links found on a company website and decide which links are useful for a business brochure. "
    "Keep about, company, careers, product, customers, and docs links. Ignore privacy, terms, login, cookie, social media, and email links. "
    "Return only valid JSON matching the requested schema."
)

link_user_prompt = (
    f"Company website: {COMPANY_URL}\n\n"
    f"Links found on the site:\n" + "\n".join(candidate_links) + "\n\n"
    "Select the links that would help write a brochure about the company. Return JSON with this shape:\n"
    '{"links": [{"type": "about page", "url": "https://..."}]}'
)

link_messages = build_messages(link_system_prompt, link_user_prompt)

print("SYSTEM:\n", link_system_prompt)
print("\nUSER:\n", link_user_prompt[:1200])
print("\nMESSAGES:\n", link_messages)

SYSTEM:
 You inspect links found on a company website and decide which links are useful for a business brochure. Keep about, company, careers, product, customers, and docs links. Ignore privacy, terms, login, cookie, social media, and email links. Return only valid JSON matching the requested schema.

USER:
 Company website: https://huggingface.co

Links found on the site:
https://huggingface.co/
https://huggingface.co/models
https://huggingface.co/datasets
https://huggingface.co/spaces
https://huggingface.co/storage
https://huggingface.co/docs
https://huggingface.co/enterprise
https://huggingface.co/pricing
https://huggingface.co/login
https://huggingface.co/join
https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro
https://huggingface.co/moonshotai/Kimi-K2.6
https://huggingface.co/Qwen/Qwen3.6-27B
https://huggingface.co/openai/privacy-filter
https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash
https://huggingface.co/spaces/smolagents/ml-intern
https://huggingface.co/spaces/r3gm/wan2-2-f

## Paso 3 manual: llamada y parsing

Si `RUN_LIVE_LLM_CALLS=False`, usamos `raw_link_json` simulado. Si lo activas, se envía `link_system_prompt` y `link_user_prompt` al modelo. Después `parse_json_output` convierte el JSON en `BrochureLinkSelection`.

In [8]:
RUN_LIVE_LLM_CALLS = True

In [9]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    link_response = llm.invoke(
        system=link_system_prompt,
        user=link_user_prompt,
        response_format={"type": "json_object"},
    )
    raw_link_json = link_response.content
else:
    raw_link_json = json.dumps({
        "links": [
            {"type": "about page", "url": "https://huggingface.co/about"},
            {"type": "models page", "url": "https://huggingface.co/models"},
            {"type": "careers page", "url": "https://huggingface.co/careers"},
        ]
    })

selected_links = parse_json_output(raw_link_json, BrochureLinkSelection)
print("RAW JSON:\n", raw_link_json)
selected_links

RAW JSON:
 {
  "links": [
    {"type": "about page", "url": "https://huggingface.co/"},
    {"type": "company", "url": "https://huggingface.co/brand"},
    {"type": "careers", "url": "https://huggingface.co/join"},
    {"type": "careers", "url": "https://apply.workable.com/huggingface/"},
    {"type": "product", "url": "https://huggingface.co/models"},
    {"type": "product", "url": "https://huggingface.co/datasets"},
    {"type": "product", "url": "https://huggingface.co/spaces"},
    {"type": "product", "url": "https://huggingface.co/storage"},
    {"type": "product", "url": "https://huggingface.co/inference/models"},
    {"type": "product", "url": "https://huggingface.co/enterprise"},
    {"type": "product", "url": "https://endpoints.huggingface.co"},
    {"type": "docs", "url": "https://huggingface.co/docs"},
    {"type": "docs", "url": "https://huggingface.co/docs/transformers"},
    {"type": "docs", "url": "https://huggingface.co/docs/diffusers"},
    {"type": "docs", "url": "htt

BrochureLinkSelection(links=[BrochureLink(type='about page', url='https://huggingface.co/'), BrochureLink(type='company', url='https://huggingface.co/brand'), BrochureLink(type='careers', url='https://huggingface.co/join'), BrochureLink(type='careers', url='https://apply.workable.com/huggingface/'), BrochureLink(type='product', url='https://huggingface.co/models'), BrochureLink(type='product', url='https://huggingface.co/datasets'), BrochureLink(type='product', url='https://huggingface.co/spaces'), BrochureLink(type='product', url='https://huggingface.co/storage'), BrochureLink(type='product', url='https://huggingface.co/inference/models'), BrochureLink(type='product', url='https://huggingface.co/enterprise'), BrochureLink(type='product', url='https://endpoints.huggingface.co'), BrochureLink(type='docs', url='https://huggingface.co/docs'), BrochureLink(type='docs', url='https://huggingface.co/docs/transformers'), BrochureLink(type='docs', url='https://huggingface.co/docs/diffusers'), B

In [20]:
print("RAW JSON:\n", raw_link_json)

RAW JSON:
 {
  "links": [
    {"type": "about page", "url": "https://huggingface.co/"},
    {"type": "company", "url": "https://huggingface.co/brand"},
    {"type": "careers", "url": "https://huggingface.co/join"},
    {"type": "careers", "url": "https://apply.workable.com/huggingface/"},
    {"type": "product", "url": "https://huggingface.co/models"},
    {"type": "product", "url": "https://huggingface.co/datasets"},
    {"type": "product", "url": "https://huggingface.co/spaces"},
    {"type": "product", "url": "https://huggingface.co/storage"},
    {"type": "product", "url": "https://huggingface.co/inference/models"},
    {"type": "product", "url": "https://huggingface.co/enterprise"},
    {"type": "product", "url": "https://endpoints.huggingface.co"},
    {"type": "docs", "url": "https://huggingface.co/docs"},
    {"type": "docs", "url": "https://huggingface.co/docs/transformers"},
    {"type": "docs", "url": "https://huggingface.co/docs/diffusers"},
    {"type": "docs", "url": "htt

In [23]:
print(selected_links)

links=[BrochureLink(type='about page', url='https://huggingface.co/'), BrochureLink(type='company', url='https://huggingface.co/brand'), BrochureLink(type='careers', url='https://huggingface.co/join'), BrochureLink(type='careers', url='https://apply.workable.com/huggingface/'), BrochureLink(type='product', url='https://huggingface.co/models'), BrochureLink(type='product', url='https://huggingface.co/datasets'), BrochureLink(type='product', url='https://huggingface.co/spaces'), BrochureLink(type='product', url='https://huggingface.co/storage'), BrochureLink(type='product', url='https://huggingface.co/inference/models'), BrochureLink(type='product', url='https://huggingface.co/enterprise'), BrochureLink(type='product', url='https://endpoints.huggingface.co'), BrochureLink(type='docs', url='https://huggingface.co/docs'), BrochureLink(type='docs', url='https://huggingface.co/docs/transformers'), BrochureLink(type='docs', url='https://huggingface.co/docs/diffusers'), BrochureLink(type='docs

## Paso 4: contenido para el brochure

`sample_company_content` existe para modo demo. Con `RUN_WEB_FETCH=True`, `fetch_website_contents` descarga contenido real desde la landing y links seleccionados. Esto sigue siendo scraping simple, no RAG.

In [24]:
RUN_WEB_FETCH = True

In [25]:
sample_company_content = """
Source URL: https://huggingface.co
Hugging Face is a collaboration platform for the machine learning community. It provides models, datasets, demos, and tools used by developers and organizations building AI products.

Source URL: https://huggingface.co/models
The model hub helps teams discover, evaluate, and share open models across text, vision, audio, and multimodal tasks.

Source URL: https://huggingface.co/careers
The company hires people interested in open-source machine learning, developer tools, infrastructure, and community-driven AI.
""".strip()

if RUN_WEB_FETCH:
    sections = [fetch_website_contents(COMPANY_URL, max_chars=8000)]
    for item in selected_links.links[:3]:
        sections.append(fetch_website_contents(item.url, max_chars=8000))
    brochure_context = "\n\n".join(sections)[:10000]
else:
    brochure_context = sample_company_content

print(brochure_context)

Source URL: https://huggingface.co

Hugging Face – The AI community building the future.
Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
about 8 hours ago
•
138k
•
2.99k
moonshotai/Kimi-K2.6
Updated
4 days ago
•
443k
•
1.09k
Qwen/Qwen3.6-27B
Updated
4 days ago
•
399k
•
896
openai/privacy-filter
Updated
5 days ago
•
47.5k
•
892
deepseek-ai/DeepSeek-V4-Flash
Updated
about 8 hours ago
•
65.7k
•
770
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
205
ML Intern
🤖
205
Get ML assistance and code suggestions from an interactive assistant
Running
on
Zero
MCP
2.4k
Wan2.2 14B Preview
🐌
2.4k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
1.03k
FireRed Image Edit 1.0 Fast
🌖
1.03k
FireRed-Ima

## Paso 5 manual: escribir el brochure

Aquí se ve claramente cómo cambia el tono. El `user_prompt` es el mismo, pero puedes cambiar `brochure_system_prompt` formal por `humorous_brochure_system_prompt`.

In [15]:
brochure_system_prompt = (
    "You write concise company brochures for prospective customers, investors, and potential recruits. "
    "Use the supplied website content only. Respond in markdown without code fences. Be factual, business-oriented, and clear."
)

humorous_brochure_system_prompt = (
    "You write company brochures for prospective customers, investors, and potential recruits. "
    "Use the supplied website content only. Respond in markdown without code fences. Keep the brochure useful, but make the tone witty and memorable."
)

brochure_user_prompt = (
    f"Company name: {COMPANY_NAME}\n"
    f"Company website: {COMPANY_URL}\n\n"
    f"Website content and selected pages:\n{brochure_context}\n\n"
    "Create a short brochure in markdown for prospective customers, investors, and potential recruits."
)

brochure_messages = build_messages(brochure_system_prompt, brochure_user_prompt)
print("FORMAL SYSTEM:\n", brochure_system_prompt)
#print("\nHUMOROUS SYSTEM:\n", humorous_brochure_system_prompt)
print("\nUSER PREVIEW:\n", brochure_user_prompt[:1200])

FORMAL SYSTEM:
 You write concise company brochures for prospective customers, investors, and potential recruits. Use the supplied website content only. Respond in markdown without code fences. Be factual, business-oriented, and clear.

USER PREVIEW:
 Company name: Hugging Face
Company website: https://huggingface.co

Website content and selected pages:
Source URL: https://huggingface.co

Hugging Face – The AI community building the future.
Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
about 8 hours ago
•
138k
•
2.99k
moonshotai/Kimi-K2.6
Updated
4 days ago
•
443k
•
1.09k
Qwen/Qwen3.6-27B
Updated
4 days ago
•
399k
•
896
openai/privacy-filter
Updated
5 days ago
•
47.5k
•
892
deepseek-ai/DeepSeek-V4-Flash
Upda

In [17]:
print(brochure_user_prompt)

Company name: Hugging Face
Company website: https://huggingface.co

Website content and selected pages:
Source URL: https://huggingface.co

Hugging Face – The AI community building the future.
Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4-Pro
Updated
about 8 hours ago
•
138k
•
2.99k
moonshotai/Kimi-K2.6
Updated
4 days ago
•
443k
•
1.09k
Qwen/Qwen3.6-27B
Updated
4 days ago
•
399k
•
896
openai/privacy-filter
Updated
5 days ago
•
47.5k
•
892
deepseek-ai/DeepSeek-V4-Flash
Updated
about 8 hours ago
•
65.7k
•
770
Browse 2M+ models
Spaces
Running
on
CPU Upgrade
205
ML Intern
🤖
205
Get ML assistance and code suggestions from an interactive assistant
Running
on
Zero
MCP
2.4k
Wan2.2 14B Preview
🐌
2.4k
generate a video from an ima

In [26]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    brochure_response = llm.invoke(system=brochure_system_prompt, user=brochure_user_prompt)
    brochure_markdown = brochure_response.content


display(Markdown(brochure_markdown))

# Hugging Face — The AI community building the future

Hugging Face is the collaboration platform for the machine learning community. The Hub is a central place to share, explore, discover, and experiment with open-source ML. We empower the next generation of ML engineers, scientists, and end users to learn, collaborate and share their work to build an open and ethical AI future together.

## For customers
- Access 45,000+ models from leading AI providers through a single, unified API with no service fees.
- Deploy on optimized Inference Endpoints or update Spaces applications to a GPU in a few clicks.
- Multi-modal capabilities: work with text, image, video, audio, or even 3D data.
- The platform supports teams and enterprises with enterprise-grade security, access controls, and dedicated support.
- Starting at $20/user/month for Team & Enterprise, with features like Single Sign-On, Regions, Priority Support, Audit Logs, Resource Groups, Private Datasets Viewer, and Inference Providers.
- Inference pricing starts at $0.60/hour for GPU.
- Join a growing community: More than 50,000 organizations are using Hugging Face.

## For investors
- A robust, multi-modal platform powering the ML pipeline: Models, Datasets, Spaces, and a unified API.
- A thriving ecosystem with a large and active open-source stack and ecosystem around Hugging Face Open Source.
- Large-scale adoption: Browse 2M+ models and 1M+ applications; browse 500k+ datasets.
- The Open Source foundation includes industry-leading libraries and tools with broad usage:
  - Transformers, Diffusers, Safetensors, Hub Python Library, Tokenizers, TRL, Transformers.js, smolagents, PEFT, Datasets, Text Generation Inference, Accelerate.
- Highly scalable compute and enterprise solutions, with a growing customer base of organizations across industries.

## For potential recruits
- Join a mission-driven company at the heart of the AI revolution, building the foundation of ML tooling with the community.
- Work on a fast-growing platform used by thousands of organizations and developers worldwide.
- Contribute to open-source projects that power the industry, including Transformers, Diffusers, Tokenizers, and more.
- Be part of a team that empowers ML engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future.

## Open Source & tooling you’ll interact with
- Transformers: State-of-the-art AI models for PyTorch
- Diffusers: State-of-the-art Diffusion models in PyTorch
- Safetensors: Safe storage/distribution of neural network weights
- Hub Python Library: Python client to interact with the Hugging Face Hub
- Tokenizers: Fast tokenizers for research and production
- TRL: Train transformers LMs with reinforcement learning
- Transformers.js: ML running directly in your browser
- smolagents: Library to build agents in Python
- PEFT: Parameter-efficient finetuning for large language models
- Datasets: Access and share datasets for ML tasks
- Text Generation Inference: Serve language models with a specialized toolkit
- Accelerate: Train PyTorch models with multi-GPU/TPU/mixed precision

## Why Hugging Face
- The AI community building the future: a platform where the ML community collaborates on models, datasets, and applications.
- Move faster with the HF Open Source stack.
- Explore all modalities and build your ML portfolio by sharing work with the world.
- Sign up and explore models, datasets, spaces, and applications today.

Visit Hugging Face to explore, collaborate, and accelerate your ML journey.

## La misma idea usando PromptRegistry

Ahora que vimos los prompts manuales, el registry se entiende mejor: solo guarda esos strings y valida variables. No decide el modelo, no llama al LLM y no parsea JSON.

In [ ]:
registry_prompt = PromptRegistry.get("brochure.select_links")
registry_user_prompt = registry_prompt.render_user(
    url=COMPANY_URL,
    links="\n".join(candidate_links),
)
registry_messages = build_messages(registry_prompt.system, registry_user_prompt)

print("Registry prompt:", registry_prompt.name)
print("\nEquivalent messages:")
registry_messages